# Netflix Content Clustering and Similarity

Cluster streaming titles and retrieve similar content from metadata and descriptions.

**Portfolio category:** Recommendation

**Data mode:** Verified demo mode

This notebook keeps labels out of fitting wherever labels exist, uses deterministic seeds,
reports unsupervised-specific diagnostics, and avoids hard-coded results.

## 1. Project setup

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

## 2. Streaming catalogue

In [ ]:
themes = {
    "crime": ["detective investigates a hidden criminal network", "forensic team solves a city mystery", "heist crew faces an impossible case"],
    "science_fiction": ["space crew explores an unknown planet", "engineer builds an artificial intelligence", "time travel changes a future colony"],
    "romance": ["two strangers build a relationship in the city", "family expectations challenge a young couple", "friends discover love during a journey"],
    "documentary": ["researchers explain climate and natural systems", "true story follows technology and society", "experts examine history through archives"],
    "comedy": ["friends navigate work and chaotic daily life", "family trip creates a chain of comic mistakes", "unlikely roommates start a ridiculous business"],
}
rows = []
for theme, descriptions in themes.items():
    for i in range(24):
        rows.append({
            "title": f"{theme.replace('_', ' ').title()} {i + 1:02d}",
            "description": rng.choice(descriptions),
            "genre": theme,
            "type": rng.choice(["Movie", "TV Show"]),
            "release_year": int(rng.integers(2000, 2026)),
        })
catalogue = pd.DataFrame(rows)
catalogue["text"] = catalogue["genre"].str.replace("_", " ") + " " + catalogue["description"]
display(catalogue.head())

## 3. Catalogue quality and coverage

In [ ]:
display(pd.Series({
    "titles": len(catalogue),
    "genres": catalogue["genre"].nunique(),
    "duplicate_titles": catalogue["title"].duplicated().sum(),
    "missing_descriptions": catalogue["description"].isna().sum(),
}).to_frame("value"))

## 4. TF-IDF representation

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
X = vectorizer.fit_transform(catalogue["text"])

## 5. Content clustering

In [ ]:
rows = []
for k in range(3, 9):
    labels = KMeans(n_clusters=k, n_init=25, random_state=RANDOM_STATE).fit_predict(X)
    rows.append({"k": k, "silhouette": silhouette_score(X, labels)})
scores = pd.DataFrame(rows)
best_k = int(scores.loc[scores["silhouette"].idxmax(), "k"])
model = KMeans(n_clusters=best_k, n_init=40, random_state=RANDOM_STATE)
catalogue["cluster"] = model.fit_predict(X)
display(scores.round(3))

## 6. Similar-title retrieval

In [ ]:
neighbours = NearestNeighbors(metric="cosine", n_neighbors=7).fit(X)

def similar_titles(title, n=6):
    index = catalogue.index[catalogue["title"].eq(title)][0]
    distances, indices = neighbours.kneighbors(X[index], n_neighbors=n + 1)
    result = catalogue.loc[indices[0][1:], ["title", "genre", "type", "cluster"]].copy()
    result["similarity"] = 1 - distances[0][1:]
    return result

seed_title = catalogue.iloc[0]["title"]
recommendations = similar_titles(seed_title)
print("Seed:", seed_title)
display(recommendations)

## 7. Evaluate neighbour relevance

In [ ]:
seed_genre = catalogue.iloc[0]["genre"]
same_genre_rate = recommendations["genre"].eq(seed_genre).mean()
display(pd.Series({
    "cluster_silhouette": silhouette_score(X, catalogue["cluster"]),
    "top_6_same_genre_rate": same_genre_rate,
    "mean_top_6_similarity": recommendations["similarity"].mean(),
}).to_frame("value"))

## 8. Cluster profiles

In [ ]:
profile = pd.crosstab(catalogue["cluster"], catalogue["genre"], normalize="index")
sns.heatmap(profile, cmap="Blues", annot=True, fmt=".2f")
plt.title("Genre composition by content cluster")
plt.tight_layout()

## 9. Key findings

Content similarity is explainable and cold-start friendly, but it should be combined with behavioural feedback for production ranking.

## 10. Interpretation and responsible use

Treat the output as exploratory evidence, not ground truth. For netflix content clustering and similarity,
validate stability on newer data, inspect edge cases, and review domain risks before
turning clusters, rankings or anomaly scores into decisions.

## 11. Next steps

- Replace demonstration data with a versioned, licensed dataset.
- Track data quality, drift and stability across repeated runs.
- Add domain-specific review before deployment.
- Package inference only after reproducibility and privacy checks pass.

All numeric results are generated at execution time; none are hard-coded.